# CoDraft — run one experiment

**Change `RUN_ID` in the next cell, then Save & Run All.** Nothing else needs editing.

On Kaggle: attach the dataset, enable Internet, and pick the **GPU T4 x2** accelerator for
every run, so all runs share the same hardware. Each run uses one GPU; a list such as
`RUN_ID = [7, 12]` runs two at once, one per card, at no extra quota.

| ID | run | fills |
|---|---|---|
| 1 / 2 | XGBoost + TF-IDF, plain / codraft | Table 2 |
| 3 / 4 | Cross-Encoder deberta, plain / codraft | Table 2, Fig. 4A (run 3) |
| 5 / 6 | Cross-Encoder bge-reranker, codraft / plain | Table 2, same backbone as ours |
| 7 | **Multi-Task, codraft — the full model** | Table 2/3, Fig. 3, Fig. 4B |
| 8 / 9 | Bi-encoder bge-m3, plain / codraft | Table 2, dense retriever |
| 10 / 11 | Cross-Encoder legal-BERT, plain / codraft | Table 2 |
| 12 | Multi-Task, plain | Table 3: w/o CoDraft Enrichment |
| 13 | Multi-Task, category | Table 3: Term + NICE heading only |
| 14 | Multi-Task, expanded | Table 3: free-text rewrite instead of Nature/Purpose |
| 15 | Multi-Task, aux_weight = 0 | Table 3: w/o Multi-Task Head |
| 16 | Multi-Task, CE loss | Table 3: w/o Rank-Aware Loss |
| 17 | Multi-Task, alpha = 0 | Table 3: focal only, no rank penalty |
| 18 | Multi-Task, aux_weight = 0 and CE loss | Table 3: the multi-task recipe alone |
| 19 | Probe of run 7, inference only | which input parts run 7 relies on |
| 20 | Hybrid MLP (Le Nir et al.), plain | Table 2: embeddings + one-hot NICE classes |
| 21 / 22 / 23 / 24 | Multi-Task, codraft by Qwen2.5-7B / Qwen3-8B / Llama-3.1-8B / Nemotron-Nano-8B | robustness: run 7 with another LLM's attributes |
| 25 | Multi-Task, codraft, binary target (Similar vs Dissimilar) | comparison with Le Nir et al. (2026) |

**Run 19 needs run 7's checkpoint.** Upload `weights/07_multi_codraft_seed42/model/` as a
Kaggle dataset and attach it next to the data; the runner finds it under `/kaggle/input`.

**Runs 21–24 need the open LLMs' data variants** (`codraft_<llm>/`, made by `scripts/llm.ipynb`).
Add those folders to the attached dataset (or attach them as a second one); the runner
finds them under `/kaggle/input`.

When the session ends, **download one file**: `/kaggle/working/codraft_runs_<IDs>.zip`. It
holds `results/` (predictions with Pair ID, metrics JSON, logits, confusion matrix),
`logs/` and `weights/` for every run of the session.

In [ ]:
# ===================== the only line to change =====================
RUN_ID = 7
# ===================================================================

# Optional:
#   RUN_ID = [7, 12]  two runs side by side, one GPU each (T4 x2)
#   SMOKE = True      tiny subset and one epoch, a few minutes: checks the pipeline
#                     before spending GPU hours. Outputs are prefixed SMOKE_.
SMOKE = False

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/CryAndRRich/CoDraft-Bench.git"
BRANCH = "new"
DATA_ROOT = "/kaggle/input/datasets/nhtquyn/codraft"   # searched for if not found here

def sh(*cmd, **kw):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)

if os.path.isdir("/kaggle/working"):
    OUT_ROOT = "/kaggle/working"
    REPO_DIR = f"{OUT_ROOT}/CoDraft-Bench"
    if os.path.isdir(REPO_DIR):
        sh("git", "-C", REPO_DIR, "pull", "--ff-only")
    else:
        sh("git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR)
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt")
else:
    # Local: run from anywhere inside the repo, with the splits in data/.
    REPO_DIR = os.getcwd()
    while REPO_DIR != "/" and not os.path.isfile(os.path.join(REPO_DIR, "runner.py")):
        REPO_DIR = os.path.dirname(REPO_DIR)
    OUT_ROOT, DATA_ROOT = REPO_DIR, None

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"repo {REPO_DIR} | branch {branch} | commit {commit}")
assert branch == BRANCH, f"expected branch {BRANCH}, got {branch}"


In [ ]:
# Each run executes in its own process pinned to one GPU, so its packages are the ones
# just installed and two runs never share a card.
from runner import launch, find_data_root
from config.runs import describe

print(describe(), "\n")
DATA_ROOT = find_data_root(DATA_ROOT)
launch(RUN_ID, data_root=DATA_ROOT, out_root=OUT_ROOT, smoke=SMOKE)

## Result

In [ ]:
import glob, json
from IPython.display import Image, display

ids = [RUN_ID] if isinstance(RUN_ID, int) else RUN_ID
for f in sorted(glob.glob(f"{OUT_ROOT}/results/*_metrics.json")):
    r = json.load(open(f))
    if r["run_id"] not in ids or r["smoke"] != SMOKE:
        continue
    print(f"\n{r['tag']}  [{r['table']}] {r['row']}")
    print(f"  Macro-F1 {r['f1_macro']*100:.2f} | QWK {r['qwk']:.4f} | MAE {r['mae']:.4f} | "
          f"adjacent {r['adjacent_acc']*100:.2f}% | severe {r['severe_rate']*100:.2f}% | n={r['n_test']}")
    print("  per-class F1: " + " / ".join(f"{c['name']} {c['f1']*100:.2f}" for c in r["per_class"]))
    print(f"  {r['train_seconds']/60:.1f} min on {r['environment']['gpu']}")
    png = f"{OUT_ROOT}/results/cm_{r['tag']}.png"
    if os.path.isfile(png):
        display(Image(png, width=480))

zips = sorted(glob.glob(f"{OUT_ROOT}/*codraft_runs_*.zip"), key=os.path.getmtime)
if zips:
    z = zips[-1]
    print(f"\nDOWNLOAD THIS: {z}  ({os.path.getsize(z) / 2**20:.1f} MB)")
else:
    print("\nno zip found; the launch cell may have failed before packaging")